In [1]:
# ================================
# 1. Import des librairies
# ================================
import pandas as pd
import h2o  # Librairie H2O
from h2o.automl import H2OAutoML  # AutoML H2O
from sklearn.datasets import load_iris

# ================================
# 2. Initialisation de H2O
# ================================
h2o.init()  # Démarre le serveur H2O local

# ================================
# 3. Chargement des données
# ================================
iris = load_iris()

# Transformer en DataFrame pandas
data = pd.DataFrame(iris.data, columns=iris.feature_names)

# Ajouter une colonne cible
data['salary'] = iris.target

# ================================
# 4. Transformation en classification binaire
# ================================
data['salary'] = (data['salary'] > 0).astype(int)

# ================================
# 5. Conversion en H2OFrame
# ================================
hf = h2o.H2OFrame(data)

# Définir la variable cible et les variables explicatives
target = 'salary'
features = hf.columns
features.remove(target)

# IMPORTANT : convertir la cible en facteur (classification)
hf[target] = hf[target].asfactor()

# ================================
# 6. Split train / test
# ================================
train, test = hf.split_frame(ratios=[0.75], seed=42)

# ================================
# 7. AutoML H2O
# ================================
aml = H2OAutoML(
    max_models=10,        # Nombre maximum de modèles testés
    seed=42,              # Reproductibilité
    verbosity="info"      # Affichage des logs
)

# Entraînement
aml.train(x=features, y=target, training_frame=train)

# ================================
# 8. Leaderboard (meilleur modèle)
# ================================
print(aml.leaderboard)

# ================================
# 9. Prédictions
# ================================
predictions = aml.leader.predict(test)

# Afficher les premières lignes
print(predictions.head())

# ================================
# 10. Sauvegarde du modèle
# ================================
model_path = h2o.save_model(model=aml.leader, path=".", force=True)
print("Modèle sauvegardé ici :", model_path)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Zulu11.86+21-CA (build 11.0.30+7-LTS, mixed mode)
  Starting server from C:\Users\User\miniconda3\envs\automl_h20_env\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\User\AppData\Local\Temp\tmp73zzfu8q
  JVM stdout: C:\Users\User\AppData\Local\Temp\tmp73zzfu8q\h2o_User_started_from_python.out
  JVM stderr: C:\Users\User\AppData\Local\Temp\tmp73zzfu8q\h2o_User_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,06 secs
H2O_cluster_timezone:,America/Toronto
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,27 days
H2O_cluster_name:,H2O_from_python_User_g77nb7
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.914 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
16:04:53.161: Project: AutoML_1_20260408_160453
16:04:53.167: 5-fold cross-validation will be used.
16:04:53.167: Setting stopping tolerance adaptively based on the training frame: 0.05
16:04:53.168: Build control seed: 42
16:04:53.170: training frame: Frame key: AutoML_1_20260408_160453_training_py_3_sid_b82f    cols: 5    rows: 115  chunks: 1    size: 1433  checksum: 6584825671741988163
16:04:53.170: validation frame: NULL
16:04:53.170: leaderboard frame: NULL
16:04:53.170: blending frame: NULL
16:04:53.170: response column: salary
16:04:53.171: fold column: null
16:04:53.171: weights column: null
16:04:53.202: AutoML: XGBoost is not available; skipping it.
16:04:53.215: Loading execution steps: [{XGBoost : [def_2 (1g, 10w), def_1 (2g, 10w), def_3 (3g, 10w), grid_1 (4g, 90w), lr_search (7g, 30w)]}, {GLM : [def_1 (1g, 10w)]}, {DRF : [def_1 (2g, 10w), XRT (3g, 10w)]}, {GBM 